In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
#1. Data loading

In [3]:
DATA_DIR = r"C:\Users\dkrok\Desktop\ml-20m"

In [4]:
import os
#the sample of 10000 rows was loaded as a representative sample to manage memory/processing time
df_movies  = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"),  nrows=10000, sep=',') 
df_ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"), nrows=10000, sep=',')

In [5]:
# a variable genres was converted to a list instead of a string
df_movies['genres'] = df_movies['genres'].str.split('|')

In [6]:
# display initial data structure
print(df_movies.head(10))

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   
5        6                         Heat (1995)   
6        7                      Sabrina (1995)   
7        8                 Tom and Huck (1995)   
8        9                 Sudden Death (1995)   
9       10                    GoldenEye (1995)   

                                              genres  
0  [Adventure, Animation, Children, Comedy, Fantasy]  
1                     [Adventure, Children, Fantasy]  
2                                  [Comedy, Romance]  
3                           [Comedy, Drama, Romance]  
4                                           [Comedy]  
5                          [Action, Crime, Thriller]  
6                                  [Comedy, Romance]  
7        

In [7]:
print(df_ratings.head(10))

   userId  movieId  rating   timestamp
0       1        2     3.5  1112486027
1       1       29     3.5  1112484676
2       1       32     3.5  1112484819
3       1       47     3.5  1112484727
4       1       50     3.5  1112484580
5       1      112     3.5  1094785740
6       1      151     4.0  1094785734
7       1      223     4.0  1112485573
8       1      253     4.0  1112484940
9       1      260     4.0  1112484826


In [8]:
#2. Preprocessing and presonalisation

In [9]:
#Split the existing ratings into training and testing sets (80/20). This allows us to check how well our model predicts "unknown" ratings.
df_train, df_test = train_test_split(df_ratings, test_size=0.2, random_state=42)

In [10]:
# Define personal ratings for User 0 (userId, movieId, rating)
my_user_id = 0
my_rated_movies = [
    (0, 48, 3.0), (0, 215, 5.0), (0, 1022, 2.0), (0, 1246, 5.0),
    (0, 1721, 2.0), (0, 2081, 4.0), (0, 3094, 3.0), (0, 8533, 5.0),
    (0, 7669, 4.0), (0, 8638, 5.0), (0, 2572, 5.0), (0, 2671, 5.0), 
    (0, 7361, 4.0), (0, 6539, 4.0), (0, 6155, 5.0), (0, 838, 1.0),
    (0, 5995, 4.0), (0, 5816, 3.0), (0, 4896, 4.0), (0, 318, 4.0)
]
# Create a DataFrame for our personal ratings and merge it with the training set
df_custom = pd.DataFrame(my_rated_movies, columns=['userId', 'movieId', 'rating'])
df_all_ratings = pd.concat([df_train, df_custom], ignore_index=True)


In [11]:
# Pivot the long-format ratings table into a 2D user-item matrix
# Rows = users, Columns = movieIds, Values = ratings (NaN where not rated)
user_item_matrix = df_all_ratings.pivot(index='userId', columns='movieId', values='rating')

In [12]:
# Filling in missing values with each user's own average rating (not 0)
# This assumes that if a user hasn't seen a movie, they'd likely give it their "average" score
user_means = user_item_matrix.mean(axis=1)
user_item_matrix = user_item_matrix.apply(lambda row: row.fillna(row.mean()), axis=1)

In [13]:
#3. Model building (matrix factorization)

In [14]:
# Initialize NMF with 20 latent factors
# n_components=20 means we discover 20 hidden "taste dimensions" (e.g., action-ness, drama level)
# init='nndsvda' uses SVD-based initialization for faster, more stable convergence
# max_iter=500 gives the optimizer enough iterations to converge fully
model = NMF(n_components=20, init='nndsvda', random_state=42, max_iter=500)
# Decompose the large matrix into two smaller ones
# fit_transform returns W - the user latent factor matrix 
W = model.fit_transform(user_item_matrix)  # shape: (users, 20); how much each user likes each of the 20 features
# model.components_ returns H - the item latent factor matrix
H = model.components_                      # shape: (20, movies); how much each movie belongs to each of the 20 features

In [15]:
# Reconstruct the matrix by multiplying W and H
# This fills every (user, movie) cell (including originally missing ones) with a predicted score
predicted_ratings = np.dot(W, H)
df_predictions = pd.DataFrame(predicted_ratings, columns=user_item_matrix.columns, index=user_item_matrix.index)

In [16]:
#4. Evaluation

In [17]:
# Compare our predictions against the hidden 'df_test' set to calculate error
actuals, preds = [], []

for _, row in df_test.iterrows():
    uid, mid, true_rating = row['userId'], row['movieId'], row['rating']
    # Only calculate if both user and movie were present in the training matrix
    if uid in df_predictions.index and mid in df_predictions.columns:
        actuals.append(true_rating)
        preds.append(df_predictions.loc[uid, mid])
        
# Compute RMSE: square root of mean squared error between actuals and predictions
rmse = np.sqrt(mean_squared_error(actuals, preds))
print(f"RMSE on test set: {rmse:.4f}")

RMSE on test set: 0.9554


In [18]:
#5. Generating recommendations (get top 20 personalised recommendations (for User 0))

In [19]:
# Extract User 0's predicted scores for all movies
user_0_preds = df_predictions.loc[0]

#Remove movies User 0 already rated - we only want to recommend unseen movies
rated_ids = [m[1] for m in my_rated_movies]
user_0_unrated = user_0_preds.drop(labels=rated_ids)

# Sort predictions from highest to lowest and grab the top 20
top_20_ids = user_0_unrated.sort_values(ascending=False).head(20)
df_top_20 = pd.DataFrame({'movieId': top_20_ids.index, 'prediction': top_20_ids.values})

# Merge with the movies dataframe to get the titles and genres for display
result = df_top_20.merge(df_movies, on='movieId')

print("\n--- TOP 20 RECOMMENDATIONS FOR YOU ---")
print(result[['title', 'genres', 'prediction']])


--- TOP 20 RECOMMENDATIONS FOR YOU ---
                                                title  \
0                             Schindler's List (1993)   
1          Life Is Beautiful (La Vita è bella) (1997)   
2                          Usual Suspects, The (1995)   
3                               Godfather, The (1972)   
4                                Fugitive, The (1993)   
5                                        Fargo (1996)   
6                                     Maverick (1994)   
7   Amelie (Fabuleux destin d'Amélie Poulain, Le) ...   
8                                Birdcage, The (1996)   
9                                    Rock, The (1996)   
10                   Silence of the Lambs, The (1991)   
11                                  Braveheart (1995)   
12                                   Toy Story (1995)   
13                                Pulp Fiction (1994)   
14                                Crimson Tide (1995)   
15                     Godfather: Part II, The (

In [20]:
# Cold-start fallback: find the most Pearson-correlated user to User 0
# if User 0's predictions look weak supplement with ratings from the most similar user

def find_similar_user(user_item_matrix, target_user=0, top_n=1):
    target = user_item_matrix.loc[target_user]
    similarities = {}
    for uid in user_item_matrix.index:
        if uid == target_user:
            continue
        other = user_item_matrix.loc[uid]
        # Only compare on movies both users have rated
        common = (target != 0) & (other != 0)
        if common.sum() < 2:  # need at least 2 common movies for meaningful correlation
            continue
        corr = np.corrcoef(target[common], other[common])[0, 1]
        if not np.isnan(corr):
            similarities[uid] = corr
    return sorted(similarities, key=similarities.get, reverse=True)[:top_n]

similar_users = find_similar_user(user_item_matrix)
print(f"Most similar user to you: {similar_users}")

# Show what that user liked, as a fallback suggestion
if similar_users:
    sim_uid = similar_users[0]
    sim_preds = df_predictions.loc[sim_uid]
    sim_top = sim_preds.drop(labels=rated_ids, errors='ignore').sort_values(ascending=False).head(5)
    df_sim_top = pd.DataFrame({'movieId': sim_top.index, 'prediction': sim_top.values})
    fallback = df_sim_top.merge(df_movies, on='movieId')
    print("\n--- FALLBACK RECOMMENDATIONS (from your most similar user) ---")
    print(fallback[['title', 'genres', 'prediction']])

Most similar user to you: [72]

--- FALLBACK RECOMMENDATIONS (from your most similar user) ---
                     title                            genres  prediction
0    Godfather, The (1972)                    [Crime, Drama]    4.393392
1      Pulp Fiction (1994)  [Comedy, Crime, Drama, Thriller]    4.309998
2   American Beauty (1999)                   [Comedy, Drama]    4.273776
3  Schindler's List (1993)                      [Drama, War]    4.266396
4        Braveheart (1995)              [Action, Drama, War]    4.262626
